In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc matplotlib
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Hello world

<details>
<summary><b>패키지 버전</b></summary>

이 페이지의 코드는 다음 요구 사항을 사용하여 개발되었습니다.
이 버전 이상을 사용하는 것을 권장합니다.

```
qiskit[all]~=2.3.0
qiskit-ibm-runtime~=0.43.1
```
</details>
이 예제는 두 부분으로 구성되어 있습니다. 먼저 간단한 양자 프로그램을 만들고 QPU(양자 처리 장치)에서 실행합니다. 실제 양자 연구에는 훨씬 더 견고한 프로그램이 필요하므로, 두 번째 섹션([대규모 Qubit로 확장하기](#scale-to-large-numbers-of-qubits))에서는 이 간단한 프로그램을 유틸리티 수준으로 확장합니다.

## 설치 및 인증
1. Qiskit을 아직 설치하지 않았다면 [빠른 시작](/guides/quick-start) 가이드에서 설치 방법을 확인하세요.

    - 양자 하드웨어에서 작업을 실행하려면 Qiskit Runtime을 설치하세요:

        ```bash
        pip install qiskit-ibm-runtime
        ```

    - Jupyter 노트북을 로컬에서 실행하기 위한 환경을 설정하세요:

        ```bash
        pip install jupyter
        ```

2. 무료 [Open Plan](/guides/plans-overview#open-plan)을 통해 양자 하드웨어에 접근하기 위한 인증을 설정하세요.

    (계정 초대 이메일을 받으셨다면, 대신 [초대된 사용자를 위한 단계](/guides/cloud-setup-invited)를 따르세요.)

    - [IBM Quantum Platform](https://quantum.cloud.ibm.com/)에서 로그인하거나 계정을 만드세요.
         > **Note:** 프록시 서버를 통해 연결하는 경우 Qiskit Runtime v0.44.0 이상을 사용해야 합니다.
    - [대시보드](https://quantum.cloud.ibm.com/)에서 API 키(*API 토큰*이라고도 함)를 생성한 다음 안전한 위치에 복사해 두세요.
    - [인스턴스](https://quantum.cloud.ibm.com/instances) 페이지로 이동하여 사용하려는 인스턴스를 찾으세요. CRN 위에 마우스를 올리고 클릭하여 복사하세요.

    - 이 코드를 사용하여 자격 증명을 로컬에 저장하세요:

        ```python
        from qiskit_ibm_runtime import QiskitRuntimeService

        QiskitRuntimeService.save_account(
        token="<your-api-key>", # Use the 44-character API_KEY you created and saved from the IBM Quantum Platform Home dashboard
        instance="<CRN>", # Optional
        )
        ```

3. 이제 Qiskit Runtime Service에 인증할 때마다 이 Python 코드를 사용할 수 있습니다:
    ```python
        from qiskit_ibm_runtime import QiskitRuntimeService

        # Run every time you need the service
        service = QiskitRuntimeService()
    ```
> **Info:** 공용 컴퓨터 또는 기타 보안이 취약한 환경을 사용하고 있다면, 인증 자격 증명을 안전하게 보호하기 위해 대신 [수동 인증 방법](/guides/cloud-setup-untrusted)을 따르세요.
## 간단한 양자 프로그램 만들기 및 실행하기
Qiskit 패턴을 사용하여 양자 프로그램을 작성하는 네 가지 단계는 다음과 같습니다:

1.  문제를 양자 네이티브 형식으로 변환합니다.

2.  Circuit와 연산자를 최적화합니다.

3.  양자 프리미티브 함수를 사용하여 실행합니다.

4.  결과를 분석합니다.

### 1단계. 문제를 양자 네이티브 형식으로 변환하기
양자 프로그램에서 *양자 Circuit*은 양자 명령어를 나타내는 네이티브 형식이며, *연산자*는 측정할 관측 가능량을 나타냅니다. Circuit를 만들 때 보통 새로운 [`QuantumCircuit`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit#quantumcircuit-class) 객체를 만든 다음 순서대로 명령어를 추가합니다.
다음 코드 셀은 *벨 상태(Bell state)*를 생성하는 Circuit를 만듭니다. 벨 상태는 두 개의 Qubit가 서로 완전히 얽혀 있는 상태입니다.

> **Note:** Qiskit SDK는 $n^{th}$ 자리가 $1 \ll n$ 또는 $2^n$ 값을 갖는 LSb 0 비트 번호 체계를 사용합니다. 자세한 내용은 [Qiskit SDK에서의 비트 순서](/guides/bit-ordering) 주제를 참조하세요.

In [ ]:
from qiskit import QuantumCircuit
from qiskit.quantum_info import SparsePauliOp
from qiskit.transpiler import generate_preset_pass_manager
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import EstimatorOptions
from qiskit_ibm_runtime import EstimatorV2 as Estimator
from matplotlib import pyplot as plt
# Uncomment the next line if you want to use a simulator:
# from qiskit_ibm_runtime.fake_provider import FakeBelemV2


# Create a new circuit with two qubits
qc = QuantumCircuit(2)

# Add a Hadamard gate to qubit 0
qc.h(0)

# Perform a controlled-X gate on qubit 1, controlled by qubit 0
qc.cx(0, 1)

# Return a drawing of the circuit using MatPlotLib ("mpl").
# These guides are written by using Jupyter notebooks, which
# display the output of the last line of each cell.
# If you're running this in a script, use `print(qc.draw())` to
# print a text drawing.
qc.draw("mpl")

<Image src="../docs/images/guides/hello-world/extracted-outputs/930ca3b6-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/hello-world/extracted-outputs/930ca3b6-0.svg)

모든 사용 가능한 연산에 대해서는 문서의 [`QuantumCircuit`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit#quantumcircuit-class)을 참조하세요.
양자 Circuit를 만들 때 실행 후 반환받을 데이터 유형도 고려해야 합니다. Qiskit은 데이터를 반환하는 두 가지 방법을 제공합니다: 측정하려는 Qubit 집합에 대한 확률 분포를 얻거나, 관측 가능량의 기댓값을 얻을 수 있습니다. [Qiskit 프리미티브](/guides/get-started-with-primitives)([3단계](#step-3-execute-using-the-quantum-primitives)에서 자세히 설명)를 사용하여 이 두 가지 방법 중 하나로 Circuit를 측정하도록 워크로드를 준비하세요.

이 예제에서는 `qiskit.quantum_info` 서브모듈을 사용하여 기댓값을 측정합니다. 연산자(양자 상태를 변경하는 동작이나 프로세스를 나타내는 데 사용되는 수학적 객체)를 사용하여 지정합니다. 다음 코드 셀은 여섯 개의 두-Qubit Pauli 연산자(`IZ`, `IX`, `ZI`, `XI`, `ZZ`, `XX`)를 생성합니다.

In [2]:
# Set up six different observables.

observables_labels = ["IZ", "IX", "ZI", "XI", "ZZ", "XX"]
observables = [SparsePauliOp(label) for label in observables_labels]

> **Note:** 여기서 `ZZ` 연산자와 같은 표기는 텐서곱 $Z\otimes Z$의 약식 표현으로, Qubit 1에서 Z를 측정하고 Qubit 0에서 Z를 함께 측정하여 Qubit 1과 Qubit 0 사이의 상관관계에 대한 정보를 얻는 것을 의미합니다. 이와 같은 기댓값은 보통 $\langle Z_1 Z_0 \rangle$으로 표기합니다.
> 
> 상태가 얽혀 있다면, $\langle Z_1 Z_0 \rangle$의 측정값은 $\langle I_1 \otimes Z_0 \rangle \langle Z_1 \otimes I_0 \rangle$의 측정값과 달라야 합니다. 위에서 설명한 Circuit로 생성된 특정 얽힘 상태의 경우, $\langle Z_1 Z_0 \rangle$의 측정값은 1이어야 하고 $\langle I_1 \otimes Z_0 \rangle \langle Z_1 \otimes I_0 \rangle$의 측정값은 0이어야 합니다.
<span id="optimize"></span>

### 2단계. Circuit와 연산자 최적화하기

디바이스에서 Circuit를 실행할 때, Circuit에 포함된 명령어 집합을 최적화하고 Circuit의 전체 깊이(대략 명령어 수)를 최소화하는 것이 중요합니다. 이를 통해 오류와 노이즈의 영향을 줄여 최상의 결과를 얻을 수 있습니다. 또한 Circuit의 명령어는 Backend 디바이스의 [ISA(Instruction Set Architecture)](/guides/transpile#instruction-set-architecture)를 준수해야 하며, 디바이스의 기본 Gate와 Qubit 연결성을 고려해야 합니다.

다음 코드는 작업을 제출할 실제 디바이스를 초기화하고, Circuit와 연산자를 해당 Backend의 ISA에 맞게 변환합니다. [자격 증명을 저장](/guides/cloud-setup)해 두셨어야 합니다.

In [ ]:
service = QiskitRuntimeService()

backend = service.least_busy(simulator=False, operational=True)

# Convert to an ISA circuit and layout-mapped observables.
pm = generate_preset_pass_manager(backend=backend, optimization_level=1)
isa_circuit = pm.run(qc)

isa_circuit.draw("mpl", idle_wires=False)

<Image src="../docs/images/guides/hello-world/extracted-outputs/9a901271-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/hello-world/extracted-outputs/9a901271-0.svg)

### 3단계. 양자 프리미티브를 사용하여 실행하기
양자 컴퓨터는 무작위 결과를 생성할 수 있으므로, 보통 Circuit를 여러 번 실행하여 출력 샘플을 수집합니다. `Estimator` 클래스를 사용하여 관측 가능량의 값을 추정할 수 있습니다. `Estimator`는 두 가지 [프리미티브](/guides/get-started-with-primitives) 중 하나이며, 다른 하나는 양자 컴퓨터에서 데이터를 얻는 데 사용할 수 있는 `Sampler`입니다. 이 객체들은 [PUB(primitive unified bloc)](/guides/primitives#sampler)을 사용하여 선택한 Circuit, 관측 가능량, 매개변수(해당하는 경우)를 실행하는 `run()` 메서드를 가지고 있습니다.

In [4]:
# Construct the Estimator instance.

estimator = Estimator(mode=backend)
estimator.options.resilience_level = 1
estimator.options.default_shots = 5000

mapped_observables = [
    observable.apply_layout(isa_circuit.layout) for observable in observables
]

# One pub, with one circuit to run against five different observables.
job = estimator.run([(isa_circuit, mapped_observables)])

# Use the job ID to retrieve your job data later
print(f">>> Job ID: {job.job_id()}")

>>> Job ID: d5k96q4jt3vs73ds5tgg


After a job is submitted, you can wait until either the job is completed within your current python instance, or use the `job_id` to retrieve the data at a later time.  (See the [section on retrieving jobs](/docs/guides/monitor-job#retrieve-job-results-at-a-later-time) for details.)

After the job completes, examine its output through the job's `result()` attribute.

In [5]:
# This is the result of the entire submission.  You submitted one Pub,
# so this contains one inner result (and some metadata of its own).
job_result = job.result()

# This is the result from our single pub, which had six observables,
# so contains information on all six.
pub_result = job.result()[0]

In [6]:
# Check there are six observables.
# If not, edit the comments in the previous cell and update this test.
assert len(pub_result.data.evs) == 6

<Admonition type="note" title="Alternative: run the example using a simulator">
  When you run your quantum program on a real device, your workload must wait in a queue before it runs. To save time, you can instead use the following code to run this small workload on the [`fake_provider`](../api/qiskit-ibm-runtime/fake-provider) with the Qiskit Runtime local testing mode. Note that this is only possible for a small circuit. When you scale up in the next section, you will need to use a real device.

  ```python

  # Use the following code instead if you want to run on a simulator:

  from qiskit_ibm_runtime.fake_provider import FakeBelemV2
  backend = FakeBelemV2()
  estimator = Estimator(backend)

  # Convert to an ISA circuit and layout-mapped observables.

  pm = generate_preset_pass_manager(backend=backend, optimization_level=1)
  isa_circuit = pm.run(qc)
  mapped_observables = [
      observable.apply_layout(isa_circuit.layout) for observable in observables
  ]

  job = estimator.run([(isa_circuit, mapped_observables)])
  result = job.result()

  # This is the result of the entire submission.  You submitted one Pub,
  # so this contains one inner result (and some metadata of its own).

  job_result = job.result()

  # This is the result from our single pub, which had five observables,
  # so contains information on all five.

  pub_result = job.result()[0]
  ```
</Admonition>

### Step 4. Analyze the results

The analyze step is typically where you might post-process your results using, for example, measurement error mitigation or zero noise extrapolation (ZNE). You might feed these results into another workflow for further analysis or prepare a plot of the key values and data. In general, this step is specific to your problem.  For this example, plot each of the expectation values that were measured for our circuit.

The expectation values and standard deviations for the observables you specified to Estimator are accessed through the job result's `PubResult.data.evs` and `PubResult.data.stds` attributes. To obtain the results from Sampler, use the `PubResult.data.meas.get_counts()` function, which will return a `dict` of measurements in the form of bitstrings as keys and counts as their corresponding values. For more information, see [Get started with Sampler.](/docs/guides/get-started-with-primitives#get-started-with-sampler)

In [ ]:
# Plot the result

values = pub_result.data.evs

errors = pub_result.data.stds

# plotting graph
plt.plot(observables_labels, values, "-o")
plt.xlabel("Observables")
plt.ylabel("Values")
plt.show()

<Image src="../docs/images/guides/hello-world/extracted-outputs/87143fcc-0.svg" alt="Output of the previous code cell" />

> **Note:** 실제 디바이스에서 양자 프로그램을 실행하면 실행 전에 대기열에서 기다려야 합니다. 시간을 절약하기 위해 Qiskit Runtime 로컬 테스트 모드에서 [`fake_provider`](../api/qiskit-ibm-runtime/fake-provider)를 사용하여 이 소규모 워크로드를 실행하는 다음 코드를 대신 사용할 수 있습니다. 이 방법은 소규모 Circuit에서만 가능하다는 점에 유의하세요. 다음 섹션에서 규모를 확장할 때는 실제 디바이스를 사용해야 합니다.
> 
>   ```python
> 
>   # Use the following code instead if you want to run on a simulator:
> 
>   from qiskit_ibm_runtime.fake_provider import FakeBelemV2
>   backend = FakeBelemV2()
>   estimator = Estimator(backend)
> 
>   # Convert to an ISA circuit and layout-mapped observables.
> 
>   pm = generate_preset_pass_manager(backend=backend, optimization_level=1)
>   isa_circuit = pm.run(qc)
>   mapped_observables = [
>       observable.apply_layout(isa_circuit.layout) for observable in observables
>   ]
> 
>   job = estimator.run([(isa_circuit, mapped_observables)])
>   result = job.result()
> 
>   # This is the result of the entire submission.  You submitted one Pub,
>   # so this contains one inner result (and some metadata of its own).
> 
>   job_result = job.result()
> 
>   # This is the result from our single pub, which had five observables,
>   # so contains information on all five.
> 
>   pub_result = job.result()[0]
>   ```
### 4단계. 결과 분석
분석 단계는 일반적으로 측정 오류 완화(measurement error mitigation)나 제로 노이즈 외삽(ZNE, zero noise extrapolation) 등을 사용하여 결과를 후처리하는 단계입니다. 이 결과를 다른 워크플로에 전달하여 추가 분석을 수행하거나, 주요 값과 데이터를 그래프로 시각화할 수도 있습니다. 일반적으로 이 단계는 문제에 따라 다르게 구성됩니다. 이 예제에서는 Circuit에서 측정된 각 기댓값을 그래프로 나타냅니다.

Estimator에 지정한 관측량에 대한 기댓값과 표준 편차는 작업 결과의 `PubResult.data.evs` 및 `PubResult.data.stds` 속성을 통해 확인할 수 있습니다. Sampler에서 결과를 얻으려면 `PubResult.data.meas.get_counts()` 함수를 사용하세요. 이 함수는 비트 문자열을 키로, 카운트를 값으로 하는 `dict` 형태의 측정 결과를 반환합니다. 자세한 내용은 [Sampler 시작하기](/guides/get-started-with-primitives#get-started-with-sampler)를 참조하세요.

In [8]:
# Make sure the results follow the claim from the previous markdown cell.
# This can happen when the device occasionally behaves strangely. If this cell
# fails, you may just need to run the notebook again.

_results = {obs: val for obs, val in zip(observables_labels, values)}
for _label in ["IZ", "IX", "ZI", "XI"]:
    assert abs(_results[_label]) < 0.2
for _label in ["XX", "ZZ"]:
    assert _results[_label] > 0.8

![이전 코드 셀의 출력](../docs/images/guides/hello-world/extracted-outputs/87143fcc-0.svg)

Qubit 0과 1에 대해 X 및 Z의 독립적인 기댓값은 모두 0인 반면, 상관 관계(`XX` 및 `ZZ`)는 1임을 확인할 수 있습니다. 이는 양자 얽힘(quantum entanglement)의 대표적인 특성입니다.

In [ ]:
def get_qc_for_n_qubit_GHZ_state(n: int) -> QuantumCircuit:
    """This function will create a qiskit.QuantumCircuit (qc) for an n-qubit GHZ state.

    Args:
        n (int): Number of qubits in the n-qubit GHZ state

    Returns:
        QuantumCircuit: Quantum circuit that generate the n-qubit GHZ state, assuming all qubits start in the 0 state
    """
    if isinstance(n, int) and n >= 2:
        qc = QuantumCircuit(n)
        qc.h(0)
        for i in range(n - 1):
            qc.cx(i, i + 1)
    else:
        raise Exception("n is not a valid input")
    return qc


# Create a new circuit with two qubits (first argument) and two classical
# bits (second argument)
n = 100
qc = get_qc_for_n_qubit_GHZ_state(n)

## 대규모 Qubit으로 확장하기
양자 컴퓨팅에서 유틸리티 규모(utility-scale) 작업은 분야의 발전을 위해 매우 중요합니다. 이러한 작업은 훨씬 더 큰 규모의 계산을 필요로 하며, 100개 이상의 Qubit과 1000개 이상의 Gate를 사용하는 Circuit을 다루게 됩니다. 이 예제는 100-Qubit GHZ 상태를 생성하고 분석하여 IBM&reg; QPU에서 유틸리티 규모 작업을 수행하는 방법을 보여줍니다. Qiskit 패턴 워크플로를 사용하며, 각 Qubit에 대한 기댓값 $\langle Z_0 Z_i \rangle$를 측정하는 것으로 마무리됩니다.

### 1단계. 문제 매핑
$n$-Qubit GHZ 상태(기본적으로 확장된 벨 상태)를 준비하는 `QuantumCircuit`을 반환하는 함수를 작성한 다음, 이 함수를 사용하여 100-Qubit GHZ 상태를 준비하고 측정할 관측량을 수집합니다.

In [ ]:
# ZZII...II, ZIZI...II, ... , ZIII...IZ
operator_strings = [
    "Z" + "I" * i + "Z" + "I" * (n - 2 - i) for i in range(n - 1)
]
print(operator_strings)
print(len(operator_strings))

operators = [SparsePauliOp(operator) for operator in operator_strings]

['ZZIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIII', 'ZIZIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIII', 'ZIIZIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIII', 'ZIIIZIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIII', 'ZIIIIZIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIII', 'ZIIIIIZIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIII', 'ZIIIIIIZIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIII', 'ZIIIIIIIZIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIII', 'ZIIIIIIIIZIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIII', 'ZIIIIIIIIIZIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIII

다음으로, 관심 있는 연산자를 매핑합니다. 이 예제에서는 Qubit 간 `ZZ` 연산자를 사용하여 거리가 멀어질수록의 동작을 살펴봅니다. 먼 Qubit 간의 기댓값이 점점 부정확해진다면, 이는 노이즈의 수준을 나타냅니다.

In [ ]:
service = QiskitRuntimeService()

backend = service.least_busy(
    simulator=False, operational=True, min_num_qubits=100
)
pm = generate_preset_pass_manager(optimization_level=1, backend=backend)

isa_circuit = pm.run(qc)
isa_operators_list = [op.apply_layout(isa_circuit.layout) for op in operators]

### Step 3. Execute on hardware

Submit the job and enable error suppression by using a technique to reduce errors called [dynamical decoupling.](../api/qiskit-ibm-runtime/options-dynamical-decoupling-options) The resilience level specifies how much resilience to build against errors. Higher levels generate more accurate results, at the expense of longer processing times.  For further explanation of the options set in the following code, see [Configure error mitigation for Qiskit Runtime.](/docs/guides/configure-error-mitigation)

In [ ]:
options = EstimatorOptions()
options.resilience_level = 1
options.dynamical_decoupling.enable = True
options.dynamical_decoupling.sequence_type = "XY4"

# Create an Estimator object
estimator = Estimator(backend, options=options)

In [13]:
# Submit the circuit to Estimator
job = estimator.run([(isa_circuit, isa_operators_list)])
job_id = job.job_id()
print(job_id)

d5k9mmqvcahs73a1ni3g


### 3단계. 하드웨어에서 실행
작업을 제출하고, [동적 디커플링(dynamical decoupling)](../api/qiskit-ibm-runtime/options-dynamical-decoupling-options)이라는 오류 억제 기법을 사용하여 오류 억제를 활성화합니다. 복원력 수준(resilience level)은 오류에 대한 복원력의 정도를 지정합니다. 수준이 높을수록 더 정확한 결과를 얻을 수 있지만, 처리 시간이 더 길어집니다. 다음 코드에서 설정된 옵션에 대한 자세한 설명은 [Qiskit Runtime의 오류 완화 설정](/guides/configure-error-mitigation)을 참조하세요.

In [ ]:
# data
data = list(range(1, len(operators) + 1))  # Distance between the Z operators
result = job.result()[0]
values = result.data.evs  # Expectation value at each Z operator.
values = [
    v / values[0] for v in values
]  # Normalize the expectation values to evaluate how they decay with distance.

# plotting graph
plt.plot(data, values, marker="o", label="100-qubit GHZ state")
plt.xlabel("Distance between qubits $i$")
plt.ylabel(r"$\langle Z_i Z_0 \rangle / \langle Z_1 Z_0 \rangle $")
plt.legend()
plt.show()

<Image src="../docs/images/guides/hello-world/extracted-outputs/de91ebd0-0.svg" alt="Output of the previous code cell" />

The previous plot shows that as the distance between qubits increases, the signal decays because of the presence of noise.

## Next steps

<Admonition type="tip" title="Recommendations">
  -   Try one of these tutorials:
      - [Ground-state energy estimation of the Heisenberg chain with VQE](/docs/tutorials/spin-chain-vqe)
      - Solve optimization problems using [QAOA](/docs/tutorials/quantum-approximate-optimization-algorithm)
      - Train [quantum kernel](/docs/tutorials/quantum-kernel-training) models for machine learning tasks
  - Find detailed installation instructions in the [Install Qiskit](/docs/guides/install-qiskit) guide.
  - If you prefer not to install Qiskit locally, read about options to use Qiskit in an [online development environment.](/docs/guides/online-lab-environments)
  - To save multiple account credentials or to specify other account options, see detailed instructions in the [Save your login credentials](/docs/guides/save-credentials#save-your-access-credentials) guide.
</Admonition>